In [4]:
import torch
import torch.nn as nn
from spin_lattices import KagomeLattice, SquareLattice1Diag, TriangleLattice
from heisenberg_hamiltonians import HeisenbergJ1J2
from loguru import logger
from slater_determinant import SlaterDeterminant, tight_binding_init
from pathlib import Path
import numpy as np
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime


def sign_overlap(ground_state, predict_signs):
    probs = ground_state**2
    return torch.dot(ground_state, predict_signs * torch.abs(ground_state)) / probs.sum()

In [5]:
lattice = KagomeLattice(2, 4)
system = HeisenbergJ1J2(lattice, J1=1, J2=1, ground_state_cache_dir=Path("groundstates"))
system.get_eigenstates(1)

ground_state_np = np.real_if_close(system.get_ground_state_in_canonical_basis())
ground_state = torch.from_numpy(ground_state_np)

2023-04-25 17:04:35.179 | DEBUG    | heisenberg_hamiltonians:__init__:427 - use_symmetries is None and lattice is in symmetries whitelist, setting use_symmetries=True, spin_inversion=1
2023-04-25 17:04:35.180 | DEBUG    | heisenberg_hamiltonians:__init__:448 - number_spins=24
2023-04-25 17:04:35.187 | DEBUG    | heisenberg_hamiltonians:__init__:458 - Symmetry group contains 16 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-04-25 17:04:35.245 | DEBUG    | heisenberg_hamiltonians:__init__:467 - Hilbert space dimension is 85662
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-04-25 17:04:35.332 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:60 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-1.0-True-1-1.pickle
2023-04-25 17:04:35.335 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:107 - Ground state energy is -42.8245991763
2023-04-25 17:04:35.337 | DEBUG    | heisenberg

In [7]:
# with torch.no_grad():
#     det.f.copy_(
#         nn.Parameter(
#             torch.randn(system.number_spins, system.number_spins, dtype=torch.float64)
#             / np.sqrt(system.number_spins)
#         )
#     )

eps_train = 0.001
test_size = 10000
epochs = 100
batch_size = 64
lr = 1e-3
scaling = 100000

for initialization_strategy in [("tight_binding", None)]: # ["orthogonal", ("tight_binding", None), ("tight_binding", "x"), ("tight_binding", "xy")]:
    if initialization_strategy == "orthogonal":
        initialization = "orthogonal"
        keep_symmetries = None
    else:
        keep_symmetries = initialization_strategy[1]
        initialization = tight_binding_init(ts=[1], keep_symmetries=keep_symmetries)

    for run in range(10):
        dataset_seed = run

        np.random.seed(dataset_seed)
        train_set_numpy = np.random.choice(
            len(system.canonical_basis.states),
            int(eps_train * len(system.canonical_basis.states)),
            replace=False,
            p=ground_state**2,
        )

        train_set = torch.from_numpy(train_set_numpy)
        logger.debug(f"{len(train_set)=}")

        target = (ground_state[train_set] > 0).double()

        rest_set_np = np.setdiff1d(np.arange(len(system.canonical_basis.states)), train_set_numpy)
        rest_probs = ground_state_np[rest_set_np] ** 2
        rest_probs /= rest_probs.sum()
        test_set = torch.from_numpy(np.random.choice(rest_set_np, test_size, replace=False))


        logger.debug(f"{run=}")
        writer = SummaryWriter(
            log_dir=(
                f"experiments/2022_04_24/{datetime.now().strftime('%Y_%m_%d_%H_%M_%S')}"
                f"_{eps_train=}_{batch_size=}_{lr=}_{initialization=}"
                f"_{scaling=}_{keep_symmetries=}"
            )
        )

        torch.manual_seed(run)
        det = SlaterDeterminant(
            system.lattice,
            system.canonical_basis,
            initialization=initialization,
            sign_cache_dir=Path("signs_cache"),
        )

        n_batches = len(train_set) // batch_size

        criterion = nn.BCELoss()

        optimizer = torch.optim.Adam(det.parameters(), lr=lr)

        epoch = 0
        logger.debug(f"{n_batches=}")
        for epoch in range(epochs):  # loop over the dataset multiple times
            i = None
            loss = None

            overlap_train = sign_overlap(ground_state[train_set], torch.sign(det(train_set) * scaling))
            overlap_test = sign_overlap(ground_state[test_set], torch.sign(det(test_set) * scaling))


            for i in range(n_batches):
                x = train_set[i * batch_size : (i + 1) * batch_size]
                y = target[i * batch_size : (i + 1) * batch_size]

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward + backward + optimize
                det_output = det(x) * scaling
                outputs = torch.sigmoid(det_output)
            
                # print(f"{det_output=}")
                # print(f"{outputs=}")
                # 1/0
                

                #        print(det(x))
                loss = criterion(outputs, y)
                loss.backward()
                #        print(det.f.grad.norm().item())

                optimizer.step()

            assert loss is not None
            writer.add_scalar("Loss/train", loss.item(), epoch)
            writer.add_scalar("Overlap/train", overlap_train.item(), epoch)
            writer.add_scalar("Overlap/test", overlap_test.item(), epoch)

            # writer.add_scalar("Det_output/std/train", det_output.std().item(), epoch)
            with torch.no_grad():
                S = (torch.svd(det.f).S)
            if epoch % 10 == 0:
                writer.add_histogram("S", S, epoch)
            writer.add_scalar("S/std/smallest", S[:system.number_spins // 2].std().item(), epoch)
            writer.add_scalar("S/std/biggest", S[system.number_spins // 2:].std().item(), epoch)
            writer.add_scalar("S/mean/smallest", S[:system.number_spins // 2].mean().item(), epoch)
            writer.add_scalar("S/mean/biggest", S[system.number_spins // 2:].mean().item(), epoch)
            
            # log.append(
            #     {
            #         "epoch": epoch,
            #         "loss": loss.item(),
            #         "overlap_train": overlap_train.item(),
            #         "overlap_test": overlap_test.item(),
            #     }
            # )
            # logger.debug(
            #     f"Epoch {epoch} loss: {loss.item():.4f} overlap_train: {overlap_train.item():.4f} "
            #     f"overlap_test: {overlap_test.item():.4f}"
            # )
            

2023-04-25 17:09:52.945 | DEBUG    | __main__:<module>:36 - len(train_set)=2704
2023-04-25 17:09:53.142 | DEBUG    | __main__:<module>:46 - run=0
/vol/tcm10/ischurov/frustrations-eda/slater_determinant.py:114: ComplexWarning: Casting complex values to real discards the imaginary part
  return f_ij.astype(np.float64)
2023-04-25 17:09:55.697 | DEBUG    | slater_determinant:__init__:160 - Using cached signs from file signs_cache/d49b2fd515a226965b09a9b91271c65e.npy
2023-04-25 17:09:55.731 | DEBUG    | __main__:<module>:70 - n_batches=42
2023-04-25 17:10:05.053 | DEBUG    | __main__:<module>:36 - len(train_set)=2704
2023-04-25 17:10:05.315 | DEBUG    | __main__:<module>:46 - run=1
2023-04-25 17:10:07.670 | DEBUG    | slater_determinant:__init__:160 - Using cached signs from file signs_cache/d49b2fd515a226965b09a9b91271c65e.npy
2023-04-25 17:10:07.709 | DEBUG    | __main__:<module>:70 - n_batches=42
2023-04-25 17:10:16.806 | DEBUG    | __main__:<module>:36 - len(train_set)=2704
2023-04-25 1

In [8]:
torch.svd(det.f)

torch.return_types.svd(
U=tensor([[-1.8185e-02,  2.3098e-01, -3.0181e-01,  2.6770e-01, -1.7139e-01,
          2.2104e-02, -5.1864e-02,  3.5590e-01,  2.8540e-01, -3.2819e-01,
          1.1233e-03,  2.0316e-01, -1.6010e-01, -1.1285e-01, -1.0915e-01,
         -2.5497e-01, -1.3739e-01, -1.1354e-01, -3.5645e-02,  5.4125e-02,
          1.3487e-01,  2.5296e-01,  2.2547e-01,  3.3112e-01],
        [-3.1706e-01, -2.5304e-02, -1.9216e-01, -3.5133e-01,  7.1122e-02,
         -1.2588e-02,  5.1986e-02, -1.6530e-01, -4.0776e-02,  4.7973e-01,
          1.4262e-01,  2.0388e-02, -3.1970e-01, -3.7873e-03,  8.8733e-02,
          2.0045e-02, -8.6767e-02, -1.4581e-01,  1.1142e-02,  1.7075e-01,
          3.3014e-01,  3.1933e-01,  2.6210e-01,  4.9425e-02],
        [-1.0652e-01, -7.3033e-02, -3.1791e-01, -1.6017e-01, -1.3903e-01,
          5.0527e-02,  1.1039e-01, -1.6969e-01, -3.7844e-02,  1.6511e-01,
         -4.1067e-01,  1.1221e-01, -2.4767e-01, -1.0689e-01, -1.3936e-01,
         -2.5600e-01,  2.1322e-01, -

In [9]:
det.f.T - det.f

tensor([[ 0.0000, -0.2160, -0.2893, -0.1043,  0.1752,  0.1103,  0.1702,  0.0113,
          0.3296,  0.3102,  0.0457, -0.1264,  0.1587, -0.0984, -0.0192, -0.1338,
         -0.0147, -0.0757,  0.0180, -0.0823, -0.0307, -0.2271, -0.1044,  0.0803],
        [ 0.2160,  0.0000, -0.0531, -0.0745,  0.0194,  0.1685, -0.2787, -0.1292,
          0.0824,  0.0133,  0.0949, -0.0600,  0.0978, -0.0756, -0.0431, -0.0310,
         -0.0985, -0.2578,  0.0614,  0.1332, -0.1680,  0.2158,  0.0202, -0.1137],
        [ 0.2893,  0.0531,  0.0000, -0.0022, -0.1106,  0.1563, -0.0250,  0.1991,
          0.0661, -0.0699, -0.0025,  0.0117, -0.0524, -0.1775, -0.1090, -0.1055,
          0.0289, -0.0156,  0.1306, -0.1433,  0.0512,  0.1137, -0.0226, -0.2413],
        [ 0.1043,  0.0745,  0.0022,  0.0000,  0.0804, -0.1829, -0.0741,  0.0241,
         -0.0640,  0.0367,  0.1179, -0.0011,  0.0381, -0.2009, -0.1615,  0.2108,
         -0.1511, -0.0089,  0.2013, -0.1883,  0.0638,  0.1659, -0.0149,  0.0908],
        [-0.1752, -0.019